# ModernBERT fine tuning
---

In [1]:
from transformers import ModernBertForQuestionAnswering, AutoTokenizer
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = '../models/answerdotai--ModernBERT-base'


model = ModernBertForQuestionAnswering.from_pretrained(model_path, attn_implementation="flash_attention_2", dtype=torch.float16).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForQuestionAnswering LOAD REPORT from: ../models/answerdotai--ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Dataset preprocessing, tokenization
---

In [ ]:
import json
import orjson

path = ''
data = []
with open(path, 'r', encoding='utf-8') as f:
    stop_f = 100
    data = []
    for line in f:
        _ = orjson.loads(line)
        data.append(_)
        stop_f -= 1

In [2]:
from datasets import load_dataset

raw_datasets = load_dataset("squad")

In [3]:
raw_datasets.keys()

dict_keys(['train', 'validation'])

In [33]:
context = raw_datasets["train"][0:1]["context"]
question = raw_datasets["train"][0:1]["question"]

inputs = tokenizer(question, context)
tokenizer.decode(inputs["input_ids"])

['[CLS]To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?[SEP]Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.[SEP]']

In [42]:
inputs = tokenizer(
    raw_datasets["train"][0:1]["question"],
    raw_datasets["train"][0:1]["context"],
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True
)

for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))

[CLS]To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?[SEP]Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is[SEP]
[CLS]To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?[SEP] front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the[SEP]
[CLS]To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?[SEP] Next to the Main Building is the Basil

In [37]:
raw_datasets['train'][0:1]

{'id': ['5733be284776f41900661182'],
 'title': ['University_of_Notre_Dame'],
 'context': ['Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.'],
 'question': ['To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?'],
 'answers': [{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}]}

In [57]:
for i, offset in enumerate(inputs['offset_mapping']):
    print(i, offset)

0 [(0, 0), (0, 2), (2, 7), (7, 11), (11, 15), (15, 22), (22, 27), (27, 37), (37, 44), (44, 47), (47, 50), (50, 52), (52, 55), (55, 57), (57, 60), (60, 63), (63, 70), (70, 71), (0, 0), (0, 4), (4, 9), (9, 15), (15, 16), (16, 20), (20, 27), (27, 31), (31, 33), (33, 42), (42, 52), (52, 53), (53, 56), (56, 58), (58, 62), (62, 67), (67, 76), (76, 78), (78, 83), (83, 88), (88, 91), (91, 93), (93, 100), (100, 107), (107, 110), (110, 114), (114, 121), (121, 126), (126, 127), (127, 139), (139, 142), (142, 148), (148, 151), (151, 155), (155, 160), (160, 169), (169, 173), (173, 180), (180, 183), (183, 184), (184, 187), (187, 189), (189, 196), (196, 203), (203, 206), (206, 213), (213, 218), (218, 223), (223, 226), (226, 232), (232, 237), (237, 241), (241, 248), (248, 250), (250, 253), (253, 256), (256, 259), (259, 262), (262, 265), (265, 268), (268, 270), (270, 275), (275, 278), (278, 282), (282, 287), (287, 296), (296, 299), (299, 303), (303, 309), (309, 312), (312, 315), (315, 319), (319, 326), 

In [48]:
inputs['overflow_to_sample_mapping']

[0, 0, 0, 0]

In [54]:
inputs.n_sequences, inputs.sequence_ids(2)

(2,
 [None,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  None,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  None])

In [56]:
answers = raw_datasets["train"][0:1]["answers"]
start_positions = []
end_positions = []

for i, offset in enumerate(inputs["offset_mapping"]):
    sample_idx = inputs["overflow_to_sample_mapping"][i]
    answer = answers[sample_idx]
    start_char = answer["answer_start"][0]
    end_char = answer["answer_start"][0] + len(answer["text"][0])
    sequence_ids = inputs.sequence_ids(i)

    # Find the start and end of the context
    idx = 0
    while sequence_ids[idx] != 1:
        idx += 1
    context_start = idx
    while sequence_ids[idx] == 1:
        idx += 1
    context_end = idx - 1

    # If the answer is not fully inside the context, label is (0, 0)
    if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
        start_positions.append(0)
        end_positions.append(0)
    else:
        # Otherwise it's the start and end token positions
        idx = context_start
        while idx <= context_end and offset[idx][0] <= start_char:
            idx += 1
        start_positions.append(idx - 1)

        idx = context_end
        while idx >= context_start and offset[idx][1] >= end_char:
            idx -= 1
        end_positions.append(idx + 1)

start_positions, end_positions

([0, 0, 75, 45], [0, 0, 82, 52])

In [ ]:
idx = 4
sample_idx = inputs["overflow_to_sample_mapping"][idx]
answer = answers[sample_idx]["text"][0]

start = start_positions[idx]
end = end_positions[idx]
labeled_answer = tokenizer.decode(inputs["input_ids"][idx][start : end + 1])

print(f"Theoretical answer: {answer}, labels give: {labeled_answer}")

Theoretical answer: a Marian place of prayer and reflection, labels give: [CLS]


In [20]:
max_length = 512
stride = 128


def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label is (0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [21]:
train_dataset = raw_datasets["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
len(raw_datasets["train"]), len(train_dataset)

(87599, 87749)

In [22]:
def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

In [23]:
validation_dataset = raw_datasets["validation"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)
len(raw_datasets["validation"]), len(validation_dataset)

(10570, 10619)

In [24]:
sub_set = raw_datasets['validation'].select(range(100))
sub_set.set_format('torch')

sub_set 

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 100
})

In [25]:
small_eval_set = raw_datasets["validation"].select(range(100))
eval_set = small_eval_set.map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [26]:
eval_set[0]

{'input_ids': [50281,
  7371,
  13946,
  2285,
  6607,
  253,
  329,
  6739,
  387,
  6053,
  19466,
  2456,
  32,
  50282,
  15705,
  19466,
  2456,
  369,
  271,
  2448,
  5842,
  2165,
  281,
  3653,
  253,
  16928,
  273,
  253,
  3313,
  15794,
  6884,
  313,
  47,
  4639,
  10,
  323,
  253,
  4104,
  2952,
  15,
  380,
  2448,
  15794,
  12651,
  313,
  34,
  6739,
  10,
  16928,
  20734,
  42613,
  16473,
  253,
  3313,
  15794,
  12651,
  313,
  47,
  6739,
  10,
  16928,
  9756,
  42422,
  2164,
  1253,
  740,
  281,
  6233,
  616,
  2626,
  6053,
  19466,
  4060,
  15,
  380,
  2165,
  369,
  4546,
  327,
  5080,
  818,
  13,
  4022,
  13,
  387,
  35925,
  434,
  21246,
  275,
  253,
  5003,
  10765,
  6912,
  14564,
  387,
  12126,
  31417,
  13,
  5002,
  15,
  1284,
  436,
  369,
  253,
  2456,
  394,
  6053,
  19466,
  13,
  253,
  9728,
  21947,
  253,
  346,
  27716,
  257,
  19054,
  3,
  342,
  2710,
  5328,
  14,
  783,
  1314,
  21823,
  13,
  347,
  973,
  347,
 

In [27]:
eval_set.column_names

['input_ids', 'attention_mask', 'offset_mapping', 'example_id']

In [28]:
sub_eval_set = eval_set.select(range(100))
print(len(sub_eval_set[0]['input_ids']))
sub_eval_set.set_format('torch')
print(type(sub_eval_set['input_ids'][0]))

sub_eval_set[0]

512
<class 'torch.Tensor'>


{'input_ids': tensor([50281,  7371, 13946,  2285,  6607,   253,   329,  6739,   387,  6053,
         19466,  2456,    32, 50282, 15705, 19466,  2456,   369,   271,  2448,
          5842,  2165,   281,  3653,   253, 16928,   273,   253,  3313, 15794,
          6884,   313,    47,  4639,    10,   323,   253,  4104,  2952,    15,
           380,  2448, 15794, 12651,   313,    34,  6739,    10, 16928, 20734,
         42613, 16473,   253,  3313, 15794, 12651,   313,    47,  6739,    10,
         16928,  9756, 42422,  2164,  1253,   740,   281,  6233,   616,  2626,
          6053, 19466,  4060,    15,   380,  2165,   369,  4546,   327,  5080,
           818,    13,  4022,    13,   387, 35925,   434, 21246,   275,   253,
          5003, 10765,  6912, 14564,   387, 12126, 31417,    13,  5002,    15,
          1284,   436,   369,   253,  2456,   394,  6053, 19466,    13,   253,
          9728, 21947,   253,   346, 27716,   257, 19054,     3,   342,  2710,
          5328,    14,   783,  1314, 21

In [ ]:
import torch

eval_set_for_model = eval_set.remove_columns(["example_id", "offset_mapping"])
print(type(eval_set_for_model))
eval_set_for_model.set_format(type="torch")
print(eval_set_for_model.format)
print(eval_set_for_model.column_names)
print(type(eval_set_for_model['input_ids']))
print(type(eval_set_for_model[0]['input_ids']))
print(eval_set_for_model[0]['input_ids'].dim())
print(eval_set_for_model[0]['input_ids'])


device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
batch = eval_set_for_model[:]
batch = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)

<class 'datasets.arrow_dataset.Dataset'>
{'type': 'torch', 'format_kwargs': {}, 'columns': ['input_ids', 'attention_mask'], 'output_all_columns': False}
['input_ids', 'attention_mask']
<class 'datasets.arrow_dataset.Column'>
<class 'torch.Tensor'>
1
tensor([50281,  7371, 13946,  2285,  6607,   253,   329,  6739,   387,  6053,
        19466,  2456,    32, 50282, 15705, 19466,  2456,   369,   271,  2448,
         5842,  2165,   281,  3653,   253, 16928,   273,   253,  3313, 15794,
         6884,   313,    47,  4639,    10,   323,   253,  4104,  2952,    15,
          380,  2448, 15794, 12651,   313,    34,  6739,    10, 16928, 20734,
        42613, 16473,   253,  3313, 15794, 12651,   313,    47,  6739,    10,
        16928,  9756, 42422,  2164,  1253,   740,   281,  6233,   616,  2626,
         6053, 19466,  4060,    15,   380,  2165,   369,  4546,   327,  5080,
          818,    13,  4022,    13,   387, 35925,   434, 21246,   275,   253,
         5003, 10765,  6912, 14564,   387, 12126

In [30]:
start_logits = outputs.start_logits.cpu().numpy()
end_logits = outputs.end_logits.cpu().numpy()

In [31]:
start_logits, end_logits

(array([[ 1.951 ,  0.947 ,  0.387 , ...,  0.44  ,  0.44  ,  0.44  ],
        [ 1.942 ,  0.9385,  0.3472, ...,  0.44  ,  0.44  ,  0.44  ],
        [ 2.014 ,  0.215 ,  0.2076, ...,  0.44  ,  0.44  ,  0.44  ],
        ...,
        [ 1.852 , -1.252 , -0.5474, ...,  0.44  ,  0.44  ,  0.44  ],
        [ 1.604 , -0.437 , -0.6245, ...,  0.44  ,  0.44  ,  0.44  ],
        [ 1.732 , -0.3323,  0.6553, ...,  0.44  ,  0.44  ,  0.44  ]],
       shape=(100, 512), dtype=float16),
 array([[-0.4314 ,  0.896  , -1.135  , ...,  0.1342 ,  0.1342 ,  0.1342 ],
        [-0.497  ,  0.8276 , -1.008  , ...,  0.1342 ,  0.1342 ,  0.1342 ],
        [-0.5728 ,  0.7227 ,  0.4258 , ...,  0.1342 ,  0.1342 ,  0.1342 ],
        ...,
        [ 0.05054,  1.036  ,  1.597  , ...,  0.1342 ,  0.1342 ,  0.1342 ],
        [-0.2169 , -0.07776,  0.5303 , ...,  0.1342 ,  0.1342 ,  0.1342 ],
        [-0.2203 ,  0.2303 ,  1.478  , ...,  0.1342 ,  0.1342 ,  0.1342 ]],
       shape=(100, 512), dtype=float16))

In [32]:
import collections

example_to_features = collections.defaultdict(list)
for idx, feature in enumerate(eval_set):
    example_to_features[feature["example_id"]].append(idx)

In [33]:
import numpy as np

n_best = 20
max_answer_length = 30
predicted_answers = []

for example in small_eval_set:
    example_id = example["id"]
    context = example["context"]
    answers = []

    for feature_index in example_to_features[example_id]:
        start_logit = start_logits[feature_index]
        end_logit = end_logits[feature_index]
        offsets = eval_set["offset_mapping"][feature_index]

        start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
        end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
        for start_index in start_indexes:
            for end_index in end_indexes:
                # Skip answers that are not fully in the context
                if offsets[start_index] is None or offsets[end_index] is None:
                    continue
                # Skip answers with a length that is either < 0 or > max_answer_length.
                if (
                    end_index < start_index
                    or end_index - start_index + 1 > max_answer_length
                ):
                    continue

                answers.append(
                    {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                )

    best_answer = max(answers, key=lambda x: x["logit_score"])
    predicted_answers.append({"id": example_id, "prediction_text": best_answer["text"]})

In [34]:
import evaluate

metric = evaluate.load("squad")

In [36]:
theoretical_answers = [
    {"id": ex["id"], "answers": ex["answers"]} for ex in small_eval_set
]

In [37]:
print(predicted_answers[0])
print(theoretical_answers[0])

{'id': '56be4db0acb8001400a502ec', 'prediction_text': ' champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was'}
{'id': '56be4db0acb8001400a502ec', 'answers': {'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'], 'answer_start': [177, 177, 177]}}


In [38]:
metric.compute(predictions=predicted_answers, references=theoretical_answers)

{'exact_match': 0.0, 'f1': 4.445254890037497}

In [39]:
from tqdm.auto import tqdm


def compute_metrics(start_logits, end_logits, features, examples):
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    for example in tqdm(examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        # Loop through all features associated with that example
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip answers that are not fully in the context
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # Skip answers with a length that is either < 0 or > max_answer_length
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        # Select the answer with the best score
        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

In [40]:
compute_metrics(start_logits, end_logits, eval_set, small_eval_set)

  0%|          | 0/100 [00:00<?, ?it/s]

{'exact_match': 0.0, 'f1': 4.445254890037497}

In [41]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

train_dataset.set_format("torch")
validation_set = validation_dataset.remove_columns(["example_id", "offset_mapping"])
validation_set.set_format("torch")

train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    collate_fn=default_data_collator,
    batch_size=8,
)
eval_dataloader = DataLoader(
    validation_set, collate_fn=default_data_collator, batch_size=8
)

In [42]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [43]:
from accelerate import Accelerator

accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [44]:
from transformers import get_scheduler

num_train_epochs = 3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [45]:
from tqdm.auto import tqdm
import torch

output_dir = f'../models/modernbert--squad-qa'

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    # Training
    model.train()
    for step, batch in enumerate(train_dataloader):
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    start_logits = []
    end_logits = []
    accelerator.print("Evaluation!")
    for batch in tqdm(eval_dataloader):
        with torch.no_grad():
            outputs = model(**batch)

        start_logits.append(accelerator.gather(outputs.start_logits).cpu().numpy())
        end_logits.append(accelerator.gather(outputs.end_logits).cpu().numpy())

    start_logits = np.concatenate(start_logits)
    end_logits = np.concatenate(end_logits)
    start_logits = start_logits[: len(validation_dataset)]
    end_logits = end_logits[: len(validation_dataset)]

    metrics = compute_metrics(
        start_logits, end_logits, validation_dataset, raw_datasets["validation"]
    )
    print(f"epoch {epoch}:", metrics)

    # Save and upload
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
    if accelerator.is_main_process:
        tokenizer.save_pretrained(output_dir)

  0%|          | 0/32907 [00:00<?, ?it/s]

Evaluation!


  0%|          | 0/1328 [00:00<?, ?it/s]

  0%|          | 0/10570 [00:00<?, ?it/s]

epoch 0: {'exact_match': 0.13245033112582782, 'f1': 0.63348797375292}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [47]:
from transformers import pipeline

# Replace this with your own checkpoint
question_answerer = pipeline("question-answering", model=output_dir)

context = """
🤗 Transformers is backed by the three most popular deep learning libraries — Jax, PyTorch and TensorFlow — with a seamless integration
between them. It's straightforward to train your models with one before loading them for inference with the other.
"""
question = "Which deep learning libraries back 🤗 Transformers?"
question_answerer(question=question, context=context)

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

{'score': 0.00946807861328125,
 'start': 94,
 'end': 149,
 'answer': ' TensorFlow — with a seamless integration\nbetween them.'}